In [1]:
# pip install pyarrow numpy
import pyarrow.parquet as pq
import numpy as np

# Path to the Parquet file (or a directory of files)
parquet_path = "../data/raw/nng_15x15.parquet"

# Open the file as a ParquetFile object – this does NOT load data yet
pf = pq.ParquetFile(parquet_path)

# Choose the column you want statistics for
col_name = "requires_search"

# Accumulators for streaming formulas
count = 0
sum_ = 0.0
sum_sq = 0.0
min_ = np.inf
max_ = -np.inf

# Iterate over row‑groups (or batches) to keep memory bounded
for rg_index in range(pf.num_row_groups):
    # Read only the target column of the current row‑group
    table = pf.read_row_group(rg_index, columns=[col_name])
    arr = table[col_name].to_numpy()          # NumPy array, still in memory for this batch

    n = arr.size
    count += n
    sum_ += arr.sum()
    sum_sq += (arr ** 2).sum()
    min_ = min(min_, arr.min())
    max_ = max(max_, arr.max())

# Final statistics
mean = sum_ / count
variance = (sum_sq - (sum_ ** 2) / count) / (count - 1)
std_dev = np.sqrt(variance)

print(f"Count   : {count}")
print(f"Mean    : {mean:.6f}")
print(f"Std Dev : {std_dev:.6f}")
print(f"Min     : {min_}")
print(f"Max     : {max_}")

Count   : 100000
Mean    : 0.053770
Std Dev : 0.225564
Min     : False
Max     : True
